In [0]:
spark.sql("""
DROP TABLE IF EXISTS ecommerce_project.bronze.orders_incremental
""")

dbutils.fs.rm(
    "/Volumes/ecommerce_project/bronze/source_files/incremental/orders/landing/",
    True
)

dbutils.fs.rm(
    "/Volumes/ecommerce_project/bronze/source_files/incremental/orders/staging/",
    True
)

dbutils.fs.rm(
    "/Volumes/ecommerce_project/bronze/source_files/incremental/orders/schema/",
    True
)

dbutils.fs.rm(
    "/Volumes/ecommerce_project/bronze/source_files/incremental/orders/checkpoint/",
    True
)

In [0]:
from pyspark.sql.functions import ntile, col, when
from pyspark.sql.window import Window

In [0]:
orders_df = (
    spark.table("ecommerce_project.bronze.orders")
    .drop("source_file", "ingestion_timestamp")
)

In [0]:
batch_window = Window.orderBy("created_at", "order_id")

In [0]:
order_with_batch = orders_df.withColumn(
    "batch_number",
    when((col("order_id") % 2 )== 0,1).otherwise(2)
)

In [0]:
batch_1 = (
    order_with_batch
    .filter(col("batch_number") == 1)
    .drop("batch_number")
)

In [0]:
batch_2 = (
    order_with_batch
    .filter(col("batch_number") == 2)
    .drop("batch_number")
)

In [0]:
batch_1.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(
    "/Volumes/ecommerce_project/bronze/source_files/incremental/orders/landing/batch_1"
)

batch_2.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv(
    "/Volumes/ecommerce_project/bronze/source_files/incremental/orders/staging/batch_2"
)

print("Batch 1 rows:", batch_1.count())
print("Batch 2 rows:", batch_2.count())

In [0]:
from pyspark.sql.functions import current_timestamp, col

landing_path = "/Volumes/ecommerce_project/bronze/source_files/incremental/orders/landing/"
schema_path = "/Volumes/ecommerce_project/bronze/source_files/incremental/orders/schema/"
checkpoint_path = "/Volumes/ecommerce_project/bronze/source_files/incremental/orders/checkpoint/"

In [0]:
#Creating streaming dataframe
incremental_orders_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", True)
    .load(landing_path)
    .withColumn("source_file", col("_metadata.file_name"))
    .withColumn("ingestion_timestamp", current_timestamp())
)


In [0]:
#write Batch 1 to a new bronze table
query = (
    incremental_orders_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("ecommerce_project.bronze.orders_incremental")
)

query.awaitTermination()

In [0]:
spark.table(
    "ecommerce_project.bronze.orders_incremental"
).count()

In [0]:
batch_2_staging_path = (
    "/Volumes/ecommerce_project/bronze/source_files/incremental/orders/staging/batch_2/"
)

batch_2_files = dbutils.fs.ls(batch_2_staging_path)

batch_2_csv = [
    file.path
    for file in batch_2_files
    if file.name.endswith(".csv")
][0]

In [0]:
dbutils.fs.cp(
    batch_2_csv,
    landing_path + "orders_batch_2.csv"
)

In [0]:
query_2 = (
    incremental_orders_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("ecommerce_project.bronze.orders_incremental")
)

query_2.awaitTermination()

In [0]:
spark.table(
    "ecommerce_project.bronze.orders_incremental"
).count()

### Verify checkpoint-based duplicate prevention

In [0]:
query_3 = (
    incremental_orders_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow = True)
    .toTable("ecommerce_project.bronze.orders_incremental")
) 

query_3.awaitTermination()


In [0]:
spark.table(
    "ecommerce_project.bronze.orders_incremental"
).count()